# DocStruct — OHR-Bench leaderboard (Colab T4)

Produces the first citable numbers for the paper: 8 tools × 3,558 questions over 3,787 pages.

**Before running:** Runtime → Change runtime type → **T4 GPU**. On CPU this is ~16 h.

Run cells top to bottom. Cell 7 is the long one (~2 h); it checkpoints to Drive and
resumes, so if Colab reclaims the session just re-run that same cell.

Relevance mode is `page` for this corpus and that is not optional — only 1.5% of
OHR-Bench spans appear verbatim in raw PDF text, so `span` scores noise.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch, sys
if not torch.cuda.is_available():
    sys.exit('NO GPU. Runtime > Change runtime type > T4 GPU, then restart from cell 1.')
print('GPU ok:', torch.cuda.get_device_name(0))

## 2. Mount Drive

The benchmark cache goes here. Free Colab reclaims sessions without warning; a
90-minute job that dies at minute 85 with the cache on local disk loses everything.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BENCH_DIR = '/content/drive/MyDrive/docstruct_bench'
CACHE_DIR = BENCH_DIR + '/.bench_cache'
!mkdir -p "{BENCH_DIR}" "{CACHE_DIR}"
print('cache ->', CACHE_DIR)

## 3. Clone + install

`unstructured-inference` is listed explicitly: `unstructured[pdf]` does not pull it,
and `partition_pdf` imports it at module load even under `strategy="fast"`.

In [ ]:
%cd /content
!git clone -b feat/paper-draft https://github.com/CandyButcher27/DocStruct
%cd /content/DocStruct
!git log --oneline -1

In [ ]:
!pip install -q -e ".[all,benchmark-heavy]" unstructured-inference pyarrow \
   llama-index-core llama-index-embeddings-huggingface

In [ ]:
# which adapters actually import. get_adapters() swallows import errors and drops
# anything whose available() is False, so the benchmark would silently skip these
# rather than fail. Read the MISSING line before the long run.
from docstruct.eval.adapters import get_adapters
want = ['docstruct','docstruct_geo','langchain','pymupdf4llm',
        'unstructured','llamaindex','llamaindex_semantic','docling']
got = get_adapters(names=want, weights=None)
print('available:', sorted(got))
print('MISSING  :', sorted(set(want) - set(got)))

## 4. Weights

In [ ]:
!mkdir -p weights
!wget -q --show-progress -P weights \
   https://huggingface.co/hantian/yolo-doclaynet/resolve/main/yolov8m-doclaynet.pt
!ls -la weights/

## 5. Corpus (~1.6 GB)

Self-fetching. Writes `data/ohrbench/*.pdf`, `data/qa/ohrbench.json` and
`reports/ohrbench_manifest.json`.

In [ ]:
!python scripts/fetch_ohrbench.py

In [ ]:
import json, glob, collections
gold = json.load(open('data/qa/ohrbench.json'))
items = gold['items'] if isinstance(gold, dict) else gold
print(len(items), 'questions  (expect 3558)')
print(len(glob.glob('data/ohrbench/*.pdf')), 'pdfs')
src = collections.Counter(i.get('evidence_source') for i in items)
print('evidence_source:', dict(src))

## 6. Time-box the two risky tools on 3 documents

Do not skip. **docling has never once run successfully** (failed locally with
`InvalidCxxCompiler` and `std::bad_alloc` — both Windows/CPU problems Linux+GPU
should not have; this is its first real test). **`llamaindex_semantic` embeds during
chunking**, so it is far slower than the other splitters.

Extrapolate the printed time by ×(3787 / pages-in-smoke) before trusting the full-run
estimate. If docling fails here, drop it from `TOOLS` in cell 7 and record the failure —
that is a reportable result about docling, not a blocker.

In [ ]:
import glob, os, shutil
os.makedirs('/content/smoke3', exist_ok=True)
for p in sorted(glob.glob('data/ohrbench/*.pdf'))[:3]:
    shutil.copy(p, '/content/smoke3/')
print(os.listdir('/content/smoke3'))

In [ ]:
%%time
!python -m docstruct.cli benchmark \
   --pdfs-dir /content/smoke3 --qa data/qa/ohrbench.json \
   --weights weights/yolov8m-doclaynet.pt \
   --tools docling,llamaindex_semantic --relevance page \
   --report-md /content/smoke.md --report-json /content/smoke.json

## 7. The real run (~2 h on T4)

Edit `TOOLS` below if cell 6 killed docling. Checkpoints per tool per document to
Drive — **if the session dies, just re-run this cell**, it resumes.

In [ ]:
TOOLS = 'docstruct,docstruct_geo,langchain,pymupdf4llm,unstructured,llamaindex,llamaindex_semantic,docling'
# drop docling here if cell 6 failed:
# TOOLS = 'docstruct,docstruct_geo,langchain,pymupdf4llm,unstructured,llamaindex,llamaindex_semantic'

!mkdir -p reports
!python -m docstruct.cli benchmark \
   --pdfs-dir data/ohrbench --qa data/qa/ohrbench.json \
   --weights weights/yolov8m-doclaynet.pt \
   --tools {TOOLS} --relevance page \
   --cache-dir "{CACHE_DIR}" \
   --report-md reports/ohr_report.md --report-json reports/ohr_results.json

## 8. Save results to Drive — run this immediately

Before anything else. A reclaimed session takes `/content` with it.

In [ ]:
!cp reports/ohr_report.md reports/ohr_results.json "{BENCH_DIR}/"
!ls -la "{BENCH_DIR}/"
from IPython.display import Markdown, display
display(Markdown(open('reports/ohr_report.md').read()))

## 9. Secondary relevance modes — **~2 h each, run only if time allows**

No relevance rule is neutral with respect to chunk size: `span` rewards large chunks,
`page` rewards small ones, `region` was designed to be size-tolerant. The paper reports
all three and states whether the ranking survives. If DocStruct wins only under `span`,
that is uncomfortable and gets reported.

**Cost:** the checkpoint stores scored results, not retrievals, so there is no cheap
re-scoring path — each mode is a full re-run. The YOLO detector cache in `CACHE_DIR` is
reused (the expensive part), but chunking, embedding and retrieval all repeat. Budget
~2 h per mode; expect a second session. Cell 7's `page` run is the one the paper needs,
these two are the robustness check.

Checkpoints are keyed per tool **and** per relevance mode, so these resume independently
and cannot inherit cell 7's page-mode numbers.

In [ ]:
for mode in ['span', 'region']:
    !python -m docstruct.cli benchmark \
       --pdfs-dir data/ohrbench --qa data/qa/ohrbench.json \
       --weights weights/yolov8m-doclaynet.pt \
       --tools {TOOLS} --relevance {mode} \
       --cache-dir "{CACHE_DIR}" \
       --report-md reports/ohr_report_{mode}.md \
       --report-json reports/ohr_results_{mode}.json
!cp reports/ohr_report_*.md reports/ohr_results_*.json "{BENCH_DIR}/"

## 10. Download everything

Results are already on Drive; this is the second copy. Commit the JSONs to the repo
locally — they are the paper's evidence.

In [ ]:
!cd reports && zip -q -r /content/ohr_results.zip ohr_report*.md ohr_results*.json
from google.colab import files
files.download('/content/ohr_results.zip')

---
## Next, back on the local machine (no GPU needed)

Analysis on the JSON, per `futureplans.md` §3:

1. **Slice by `evidence_source`** (text 2,666 / table 847 / equation 45). If DocStruct
   loses specifically on table or equation questions, that is the measurement that
   justifies expanding the 5-label set. If it does not, do not expand it.
2. **Compare the three relevance modes** from cells 7 and 9 — does the ranking survive?
3. **Quantify the back-matter penalty.** DocStruct drops references by design, so under
   `page` relevance any question whose evidence page is in the back matter is
   structurally unreachable for us and reachable for everyone else. On the smoke,
   DocStruct's chunks stopped at page 11 of a 15-page paper while every other tool
   reached 14. Measure this before concluding anything from the page-mode numbers.

FinanceBench is a separate session (~15,000 pages, ~4× this run) and needs
`--relevance region`.